# 🤖 SLM Fine-Tuning: Coding Assistant with Tool Calling

*Prepared by **Jude Teves***.<br>
*Adapted from Unsloth's documentation*

---

**Model:** `unsloth/Llama-3.2-1B-Instruct` · **GPU:** T4 (free Colab) · **Training time:** ~3 min

This notebook fine-tunes Llama 3.2 1B to:
- Answer only coding questions (guardrail)
- Output structured `<tool_call>` blocks when it needs to look something up or run code
- Politely refuse off-topic requests

The trained LoRA adapter is saved to `coding_assistant_lora/` for use in the inference notebook.

## 1. Install

In [1]:
%%capture
!pip install unsloth
!pip install trl==0.15.2 transformers==4.51.3 datasets

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/SLMDev/'
print(f'Drive mounted. Working path: {DRIVE_PATH}')

Mounted at /content/drive
Drive mounted. Working path: /content/drive/MyDrive/SLMDev/


## 3. Load Base Model (1B, 4-bit)

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None          # auto: float16 on T4
load_in_4bit = True   # saves VRAM, required on free T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print('Base model loaded.')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth_zoo/__init__.py:405: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Base model loaded.


## 4. LoRA Adapter Setup

LoRA trains only ~24M parameters instead of the full 1B. The base model weights are frozen — nothing is destructively overwritten. The adapter is saved separately and can be swapped or rolled back anytime.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                    # adapter rank — higher = more expressive, slower
    lora_alpha = 16,           # scaling factor, usually matches r
    lora_dropout = 0,          # 0 is Unsloth's optimized default
    bias = 'none',
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)
print('LoRA adapter attached.')

Unsloth 2026.6.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


LoRA adapter attached.


## 5. Training Dataset

28 hand-crafted examples in three categories:
- **Direct answers** — coding questions the model answers immediately
- **Tool calls** — questions that trigger a `<tool_call>` block
- **Refusals** — off-topic questions the model declines

Each example includes a system prompt that defines the assistant's identity and constraints.

In [5]:
import json

# Load dataset from Google Drive — edit dataset.json to add/change examples
with open(DRIVE_PATH + 'dataset.json') as f:
    dataset_raw = json.load(f)

print(f'Loaded {len(dataset_raw)} training examples')

# Count by category
tool_calls = [ex for ex in dataset_raw if '<tool_call>' in ex['conversations'][-1]['content']]
refusals   = [ex for ex in dataset_raw if 'coding assistant' in ex['conversations'][-1]['content'].lower()
               or 'outside what' in ex['conversations'][-1]['content'].lower()
               or 'only answer' in ex['conversations'][-1]['content'].lower()
               or "only handle" in ex['conversations'][-1]['content'].lower()
               or "strictly" in ex['conversations'][-1]['content'].lower()
               or "outside" in ex['conversations'][-1]['content'].lower()]
directs    = [ex for ex in dataset_raw if ex not in tool_calls and ex not in refusals]
print(f'  Direct answers : {len(directs)}')
print(f'  Tool calls     : {len(tool_calls)}')
print(f'  Refusals       : {len(refusals)}')


Loaded 31 training examples
  Direct answers : 11
  Tool calls     : 16
  Refusals       : 4


### Preview — one example from each category

In [6]:
PREVIEW = [
    ('Direct answer', 0),
    ('Tool call',     10),
    ('Refusal',       23),
]

for label, idx in PREVIEW:
    ex = dataset_raw[idx]
    user      = ex['conversations'][1]['content']
    assistant = ex['conversations'][2]['content']
    print(f'\n{'─'*60}')
    print(f'[{label}]')
    print(f'  USER      : {user[:120]}')
    print(f'  ASSISTANT : {assistant[:200]}')



────────────────────────────────────────────────────────────
[Direct answer]
  USER      : How do I reverse a list in Python?
  ASSISTANT : Use slicing: `my_list[::-1]` — returns a new reversed list. For in-place reversal: `my_list.reverse()`. Use slicing when you need the original untouched.

────────────────────────────────────────────────────────────
[Tool call]
  USER      : I'm getting: ModuleNotFoundError: No module named 'pandas'
  ASSISTANT : Let me look that up.

<tool_call>
{"name": "lookup_error", "arguments": {"error_message": "ModuleNotFoundError: No module named 'pandas'"}}
</tool_call>

────────────────────────────────────────────────────────────
[Refusal]
  USER      : Can you calculate the sum of all even numbers from 1 to 100?
  ASSISTANT : Sure, let me write and run that.

<tool_call>
{"name": "code_and_run", "arguments": {"task_description": "calculate the sum of all even numbers from 1 to 100"}}
</tool_call>


## 6. Format Dataset for Training

In [7]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')

def format_examples(examples):
    texts = [
        tokenizer.apply_chat_template(ex['conversations'], tokenize=False, add_generation_prompt=False)
        for ex in examples['conversations']
    ]
    return {'text': texts}

# Wrap in HuggingFace Dataset
hf_dataset = Dataset.from_list([{'conversations': ex['conversations']} for ex in dataset_raw])
hf_dataset = hf_dataset.map(lambda x: {'text': tokenizer.apply_chat_template(
    x['conversations'], tokenize=False, add_generation_prompt=False
)}, batched=False)

print(f'Formatted {len(hf_dataset)} training examples.')
print('\nSample (first 300 chars):')
print(hf_dataset[0]['text'][:300])

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Formatted 31 training examples.

Sample (first 300 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

You are a coding assistant. You only answer programming and software development questions. For anything unrelated to code or software, politely decline. When you need to look


## 7. Train

`max_steps = 60` → ~3 minutes on T4. `train_on_responses_only` masks user/system tokens so the model only learns from assistant outputs — a built-in guardrail that prevents it from learning to 'act like a user'.

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 42,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)

# Only train on assistant responses (guardrail)
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|start_header_id|>user<|end_header_id|>\n\n',
    response_part = '<|start_header_id|>assistant<|end_header_id|>\n\n',
)

print('Starting training...')
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/31 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 31 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss
10,1.619300
20,0.656500
30,0.222400
40,0.043800
50,0.008700
60,0.004600


TrainOutput(global_step=60, training_loss=0.42587330856670935, metrics={'train_runtime': 55.3955, 'train_samples_per_second': 8.665, 'train_steps_per_second': 1.083, 'total_flos': 437938170900480.0, 'train_loss': 0.42587330856670935})

## 8. Save LoRA Adapter to Drive

In [9]:
SAVE_PATH = DRIVE_PATH + 'coding_assistant_lora'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Adapter saved to {SAVE_PATH}')

Adapter saved to /content/drive/MyDrive/SLMDev/coding_assistant_lora


## ✅ Done

The LoRA adapter is saved to `MyDrive/SLMDev/coding_assistant_lora/`. Open **02_inference.ipynb** to run the base vs. fine-tuned comparison.